# VehiAlpes — Capa Bronze

MINE-4214 · Taller 1 · Punto 2 (ELT con arquitectura Medallón)

## Objetivo de la capa

Dejar los tres archivos fuente en Delta **sin depurar**, tal como llegaron, más los
metadatos que permiten auditar de dónde salió cada fila.

## Decisiones y su justificación

**1. Todas las columnas se leen como `string`.**
Es la decisión más importante de esta capa. `fecha_inicio` trae cuatro formatos
distintos (ISO 55,5%, `DD/MM/AAAA` 19,2%, `MM-DD-AAAA` 14,8% y `DD-mmm-AAAA` con mes en español 10,5%).
Si dejáramos que Spark infiriera el esquema, las filas no parseables llegarían como
`null` y **perderíamos el dato de origen de forma irrecuperable**. Leyendo todo como
texto, bronze conserva el 100% de lo que entregó la fuente y el casteo se vuelve una
transformación auditable en silver.

**2. No se deduplica, no se corrige, no se filtra.**
Los 60 registros con `id_transaccion >= 5000` y los 16 vehículos con tarifas fuera
de escala entran completos. Bronze es la única capa que puede responder "¿qué nos
mandaron exactamente?", y si limpiamos aquí perdemos la línea base contra la cual
medir la calidad.

**3. Se agregan metadatos de ingesta.**
`_archivo_origen`, `_fecha_ingesta` y `_lote_ingesta` permiten reprocesar un lote
específico y dan trazabilidad de linaje.

**4. Carga incremental con Auto Loader.**
Aunque la muestra es un archivo único, se usa `cloudFiles` porque el enunciado
describe un CRM operativo que produce datos continuamente. Auto Loader lleva el
control de archivos ya procesados en el checkpoint, así que una re-ejecución no
duplica. El costo de plantearlo así desde el inicio es cero y evita reescribir el
pipeline cuando se conecte la fuente real.

In [0]:
from pyspark.sql import functions as F

CATALOGO = "vehialpes"
VOLUMEN = f"/Volumes/{CATALOGO}/landing/archivos"
RUTA_LANDING = f"{VOLUMEN}/crm"
RUTA_CHECKPOINT = f"{VOLUMEN}/_checkpoints"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for capa in ["landing", "bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{capa}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.landing.archivos")

# Subcarpeta por entidad: Auto Loader vigila un directorio por tabla
for entidad in ["carros", "clientes", "transacciones"]:
    dbutils.fs.mkdirs(f"{RUTA_LANDING}/{entidad}")

print(f"Sube cada CSV a su carpeta dentro de {RUTA_LANDING}/")

Sube cada CSV a su carpeta dentro de /Volumes/vehialpes/landing/archivos/crm/


## Columnas esperadas (validación, no esquema posicional)

### Decisión: NO se declara un esquema posicional

Es tentador declarar un `StructType` con las columnas en orden. **Es un error
grave con estos archivos.** Cuando se entrega un esquema explícito, Spark mapea las
columnas **por posición e ignora los nombres del encabezado**. El orden real de
`carros.csv` es `... tipo_combustible, fecha_ingreso_concesionario,
costo_mantenimiento_km, color ...` y el de `transacciones.csv` es
`... nombre_sucursal, ciudad, nombre_proveedor ...`. Cualquier suposición sobre ese
orden desplaza los valores en silencio: `color` termina con "Gasolina" y
`tipo_combustible` con una fecha. Las fechas quedan nulas, el join contra las
vigencias del vehículo no encuentra nada y todo el pipeline produce ceros sin
lanzar un solo error.

La alternativa es leer por nombre de columna. Auto Loader infiere los nombres del
encabezado y, con `cloudFiles.inferColumnTypes` en `false`, entrega **todas las
columnas como texto**, que es justo lo que pide la decisión 1. El orden deja de
importar y un reordenamiento en la fuente no rompe nada.

El conjunto de columnas esperadas se usa solo para **validar**: si falta una o
aparece una nueva, la carga lo reporta en lugar de continuar con datos corridos.

### Decisión: se normalizan los nombres de columna

Los tres archivos traen BOM (marca de orden de bytes) al inicio, así que la primera
columna llega nombrada `\ufeffplaca` en lugar de `placa`. Se limpia con una
normalización que quita el BOM y los espacios.

In [0]:
COLUMNAS_ESPERADAS = {
    "carros": {
        "placa", "marca", "modelo", "anio", "tipo_combustible",
        "fecha_ingreso_concesionario", "costo_mantenimiento_km", "color",
        "costo_alquiler_dia", "valor_seguro", "fecha_actualizacion",
    },
    "clientes": {"id_cliente", "numero_documento", "nombres", "apellidos"},
    "transacciones": {
        "id_transaccion", "tipo_transaccion", "fecha_inicio", "fecha_fin", "placa",
        "id_cliente", "nombre_sucursal", "ciudad", "nombre_proveedor", "metodo_pago",
        "km_transaccion", "kms_recorridos", "valor_total", "valor_seguro",
    },
}


def normalizar_nombre(nombre):
    """Quita el BOM y los espacios del nombre de columna."""
    return nombre.replace("\ufeff", "").replace("\xef\xbb\xbf", "").strip()

## Ingesta

`rescuedDataColumn` captura cualquier columna que aparezca en el archivo y no esté
en el esquema declarado. Si mañana el CRM agrega un campo, no se pierde: queda en
`_rescued_data` y se detecta en las validaciones en lugar de fallar silenciosamente.

In [0]:
def ingestar_bronze(entidad):
    """Devuelve la StreamingQuery para poder esperarla. `trigger(availableNow=True)`
    **no bloquea**: arranca el stream y devuelve el control de inmediato. Sin un
    `awaitTermination` explícito, la celda de validación puede correr antes de que
    termine la escritura y reportar cero filas en la tabla más grande."""
    lectura = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{RUTA_CHECKPOINT}/{entidad}/schema")
        .option("cloudFiles.inferColumnTypes", "false")  # todo como texto
        .option("header", "true")
        .option("encoding", "UTF-8")
        .option("rescuedDataColumn", "_rescued_data")
        .load(f"{RUTA_LANDING}/{entidad}"))

    renombradas = [F.col(f"`{c}`").alias(normalizar_nombre(c)) for c in lectura.columns]

    return (lectura.select(*renombradas)
        .withColumn("_archivo_origen", F.col("_metadata.file_path"))
        .withColumn("_fecha_ingesta", F.current_timestamp())
        .withColumn("_lote_ingesta", F.date_format(F.current_timestamp(), "yyyyMMddHHmm"))
        .writeStream
        .format("delta")
        .option("checkpointLocation", f"{RUTA_CHECKPOINT}/{entidad}/commits")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{CATALOGO}.bronze.{entidad}"))


consultas = {e: ingestar_bronze(e) for e in COLUMNAS_ESPERADAS}

# Esperar a que cada stream termine antes de validar
for entidad, consulta in consultas.items():
    consulta.awaitTermination()
    print(f"{entidad}: ingesta terminada")

carros: ingesta terminada
clientes: ingesta terminada
transacciones: ingesta terminada


## Validaciones de la capa

Bronze no corrige nada, pero sí **verifica que no perdió nada**. Estas tres
comprobaciones son la puerta de entrada a silver: si alguna falla, el problema está
en la ingesta y no tiene sentido seguir.

In [0]:
ESPERADO = {"carros": 1379, "clientes": 1560, "transacciones": 5059}

for entidad, filas_esperadas in ESPERADO.items():
    df = spark.table(f"{CATALOGO}.bronze.{entidad}")
    total = df.count()
    rescatadas = df.filter(F.col("_rescued_data").isNotNull()).count()
    columnas = {c for c in df.columns if not c.startswith("_")}
    faltantes = COLUMNAS_ESPERADAS[entidad] - columnas
    nuevas = columnas - COLUMNAS_ESPERADAS[entidad]
    print(f"{entidad}: {total} filas (esperadas {filas_esperadas}) · rescatadas {rescatadas}")
    assert total == filas_esperadas, f"{entidad}: conteo no coincide con la fuente"
    assert rescatadas == 0, f"{entidad}: hay valores que no encajan en su columna"
    assert not faltantes, f"{entidad}: faltan columnas {faltantes}"
    if nuevas:
        print(f"  aviso: columnas nuevas en la fuente: {nuevas}")

carros: 1379 filas (esperadas 1379) · rescatadas 0
clientes: 1560 filas (esperadas 1560) · rescatadas 0
transacciones: 5059 filas (esperadas 5059) · rescatadas 0


## Optimización

Las tablas son pequeñas (la mayor tiene ~5000 filas), así que no se particionan:
particionar por fecha generaría cientos de archivos diminutos y empeoraría el
desempeño. Se aplica `OPTIMIZE` para compactar y se deja el clustering por la
columna de acceso más frecuente.

In [0]:
for entidad in COLUMNAS_ESPERADAS:
    spark.sql(f"OPTIMIZE {CATALOGO}.bronze.{entidad}")